In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- medmodels_propensity_score_prop_score_migration ---
def _calculate_propensity(x_tr, y_tr, tr_arr, ctrl_arr, hyperparam=None, metric="logit"):
    return np.array([0.6, 0.7, 0.8]), np.array([0.3, 0.4, 0.5])

def _nearest_neighbor_pd(treated, control, metric="absolute", covariates=None):
    return control[["patient_id", *(covariates or [])]].head(2).copy()

def _nearest_neighbor_pl(treated, control, metric="absolute", covariates=None):
    return control.select(["patient_id", *(covariates or [])]).head(2)

FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CALCULATE_PROPENSITY = _calculate_propensity
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES = ["age", "bmi", "gender"]
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_HYPERPARAM = None
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_MODEL = "logit"
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN = np.array([[1., 4.], [2., 5.], [3., 6.]])
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN = np.array([0, 1, 0])
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PD = pd.DataFrame({"patient_id": ["C1", "C2", "C3"], "age": [45, 60, 35], "bmi": [25.0, 28.5, 22.1], "gender": [0, 1, 0]})
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PD = pd.DataFrame({"patient_id": ["T1", "T2", "T3"], "age": [50, 58, 40], "bmi": [24.0, 29.5, 23.1], "gender": [1, 1, 0]})
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PL = pl.from_pandas(FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PD)
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PL = pl.from_pandas(FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PD)
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_NEAREST_NEIGHBOR = _nearest_neighbor_pd
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PD
FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PD

# --- medmodels_propensity_score_signature_migration ---
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_CONTROL_SET_PD = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PD
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_TREATED_SET_PD = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PD
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_CONTROL_SET_PL = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PL
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_TREATED_SET_PL = FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PL
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_CONTROL_SET = FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_CONTROL_SET_PD
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_COVARIATES = ["age", "bmi", "gender"]
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_HYPERPARAM = None
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_MODEL = "logit"
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_RUN_PROPENSITY_SCORE = lambda treated, control, model="logit", hyperparam=None, covariates=None: treated
FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_TREATED_SET = FIX_MEDMODELS_PROPENSITY_SCORE_SIGNATURE_MIGRATION_TREATED_SET_PD

try:
    pd.Index.__class_getitem__
except AttributeError:
    pd.Index.__class_getitem__ = classmethod(lambda cls, item: cls)

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_medmodels_propensity_score_prop_score_migration(calculate_propensity, covariates, hyperparam, model, nearest_neighbor, x_train, y_train, control_set, treated_set):
    treated_array = treated_set[covariates].to_numpy().astype(float)
    control_array = control_set[covariates].to_numpy().astype(float)

    treated_prop, control_prop = calculate_propensity(
            x_train,
            y_train,
            treated_array,
            control_array,
            hyperparam=hyperparam,
            metric=model,
        )

    # Add propensity score to the original data
    treated_set["Prop. score"] = treated_prop
    control_set["Prop. score"] = control_prop

    matched_control = nearest_neighbor(
        treated_set, control_set, metric="absolute", covariates=["Prop. score"]
    )

    matched_control.pop("Prop. score")
    treated_set.pop("Prop. score")
    control_set.pop("Prop. score")
    return matched_control

def before_medmodels_propensity_score_signature_migration(control_set, covariates, hyperparam, model, run_propensity_score, treated_set):
    from typing import Any, Dict, List, Optional, Tuple, Union

    import numpy as np
    import pandas as pd
    ...

    def run_propensity_score(
    treated_set: pd.DataFrame,
    control_set: pd.DataFrame,
    model: str = "logit",
    hyperparam: Optional[Any] = None,
    covariates: Optional[Union[List[str], pd.Index[str]]] = None,
    ) -> pd.DataFrame:
        pass
    return run_propensity_score

In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_medmodels_propensity_score_prop_score_migration(calculate_propensity, covariates, hyperparam, model, nearest_neighbor, x_train, y_train, control_set, treated_set):

    treated_array = treated_set.select(covariates).to_numpy().astype(float)
    control_array = control_set.select(covariates).to_numpy().astype(float)

    treated_prop, control_prop = calculate_propensity(
        x_train,
        y_train,
        treated_array,
        control_array,
        hyperparam=hyperparam,
        metric=model,
    )

    # Add propensity score to the original data
    treated_set = treated_set.with_columns(pl.Series("Prop. score", treated_prop))
    control_set = control_set.with_columns(pl.Series("Prop. score", control_prop))

    matched_control = nearest_neighbor(
        treated_set, control_set, metric="absolute", covariates=["Prop. score"]
    )

    matched_control = matched_control.drop("Prop. score")
    treated_set = treated_set.drop("Prop. score")
    control_set = control_set.drop("Prop. score")
    return control_set

def gen_medmodels_propensity_score_signature_migration(control_set, covariates, hyperparam, model, run_propensity_score, treated_set):
    from typing import Any, Dict, List, Optional, Tuple, Union

    import numpy as np
    ...

    def run_propensity_score(
    treated_set: pl.DataFrame,
    control_set: pl.DataFrame,
    model: str = "logit",
    hyperparam: Optional[Any] = None,
    covariates: Optional[Union[List[str], pl.Series]] = None,
    ) -> pl.DataFrame:
        pass
    return run_propensity_score

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: medmodels_propensity_score_prop_score_migration ===

def _fresh_propensity_pd():
    return (
        FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PD.copy(),
        FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PD.copy(),
    )

def _fresh_propensity_pl():
    return (
        FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_CONTROL_SET_PL.clone(),
        FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_TREATED_SET_PL.clone(),
    )

try:
    _control, _treated = _fresh_propensity_pl()
    _r = gen_medmodels_propensity_score_prop_score_migration(_calculate_propensity, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pl, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control, _treated)
    print("✅ L1 smoke gen_medmodels_propensity_score_prop_score_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_medmodels_propensity_score_prop_score_migration: {type(_e).__name__}: {_e}")

try:
    _control, _treated = _fresh_propensity_pd()
    _rb = before_medmodels_propensity_score_prop_score_migration(_calculate_propensity, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pd, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control, _treated)
    print("✅ L1 smoke before_medmodels_propensity_score_prop_score_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_medmodels_propensity_score_prop_score_migration: {type(_e).__name__}: {_e}")

try:
    _control_pd, _treated_pd = _fresh_propensity_pd()
    _control_pl, _treated_pl = _fresh_propensity_pl()
    _rb = before_medmodels_propensity_score_prop_score_migration(_calculate_propensity, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pd, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control_pd, _treated_pd)
    _rg = gen_medmodels_propensity_score_prop_score_migration(_calculate_propensity, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pl, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control_pl, _treated_pl)
    compare(_rb, _rg, "medmodels_propensity_score_prop_score_migration")
except Exception as _e:
    print(f"❌ L2 equivalence medmodels_propensity_score_prop_score_migration: setup error — {type(_e).__name__}: {_e}")

# AUDIT-197: compare alternate-score output rather than merely running generated code.
try:
    _calc = lambda x_tr, y_tr, tr_arr, ctrl_arr, hyperparam=None, metric="logit": (np.array([0.1, 0.2, 0.3]), np.array([0.4, 0.5, 0.6]))
    _control_pd, _treated_pd = _fresh_propensity_pd()
    _control_pl, _treated_pl = _fresh_propensity_pl()
    _rb = before_medmodels_propensity_score_prop_score_migration(_calc, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pd, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control_pd, _treated_pd)
    _rg = gen_medmodels_propensity_score_prop_score_migration(_calc, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_COVARIATES, None, "logit", _nearest_neighbor_pl, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_X_TRAIN, FIX_MEDMODELS_PROPENSITY_SCORE_PROP_SCORE_MIGRATION_Y_TRAIN, _control_pl, _treated_pl)
    compare(_rb, _rg, "L3 edge medmodels_propensity_score_prop_score_migration alternate scores", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge medmodels_propensity_score_prop_score_migration alternate scores: {type(_e).__name__}: {_e}")
